<a href="https://colab.research.google.com/github/SrijanKumar123/flyrank-ml-internship/blob/main/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SrijanKumar123/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
%pip -q install duckdb
import duckdb
from google.colab import userdata

token = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.execute(f"""
    CREATE OR REPLACE SECRET hf (
        TYPE huggingface,
        TOKEN '{token}'
    )
""")

In [ ]:
REL = """
read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

In [ ]:
%pip -q install -U duckdb huggingface_hub

import duckdb
from google.colab import userdata
from huggingface_hub import whoami

token = userdata.get("HF_TOKEN")

print("Token found:", token is not None)
print("Logged in as:", whoami(token=token)["name"])

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 89.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 kB 49.6 MB/s eta 0:00:00
Token found: True
Logged in as: srijan317


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

## 1. Two Paper Findings & My Methodology Questions

### Finding #1: The Freshness Multiplier on Mature Pages (Finding #4)
* **Claim from Paper:** Content older than 365 days that was refreshed within the last 30 days demonstrated a 3.2x health score boost and a 57x impression boost compared to un-refreshed stale pages.
* **Methodology Question:** *What criteria governed which pages received a refresh? If content managers selectively refreshed historically high-performing assets (survivor bias), does the measured 57x impression lift reflect the quality of the refresh or the baseline authority of the selected pages?*

---

### Finding #2: AI-Generated Content Penalty Debunk (Finding #5 / Myth #5)
* **Claim from Paper:** Age-controlled model cohorts show no blanket search penalty tied solely to AI usage across 340k+ pages authored by 5 AI models.
* **Methodology Question:** *How were the age-controlled cohorts split across domains? Were records from individual client domains partitioned strictly using a grouped split, or could domain authority effects leak across cohorts and obscure page-level AI penalties?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

* **Before (Random Split):** Evaluating with `train_test_split` allows records from the same domain (`client_hash_id`) to land in both training and test sets. This causes client-level data leakage and inflates performance metrics.
* **After (Grouped Split):** Grouping by `client_hash_id` via `GroupKFold` guarantees that entire client portfolios stay isolated in either train or validation folds, providing an honest measure of how well the model generalizes to unseen domains[cite: 2].

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import numpy as np
import pandas as pd

dataset = con.sql(f""" SELECT client_hash_id, gsc_avg_position, gsc_impressions, ga4_pageviews, ga4_sessions, gsc_clicks
FROM {REL} """).df()
dataset["ctr"] = dataset["gsc_clicks"] / dataset["gsc_impressions"].replace(
    0, np.nan
)

dataset["is_striking_distance"] = (
    (dataset["gsc_avg_position"] > 3) &
     (dataset["gsc_avg_position"] <= 20) &
 (dataset["gsc_impressions"] >= 500)
 ).astype(int)

dataset["ctr"] = dataset["ctr"].fillna(0)
dataset["ga4_pageviews"] = dataset["ga4_pageviews"].fillna(0)
dataset["ga4_sessions"] = dataset["ga4_sessions"].fillna(0)
dataset["gsc_avg_position"] = dataset["gsc_avg_position"].fillna(100.0)

y = dataset["is_striking_distance"]
X = dataset[["gsc_avg_position", "gsc_impressions", "ctr",  "ga4_pageviews", "ga4_sessions"]]

In [ ]:
from sklearn.model_selection import GroupKFold

groups = dataset["client_hash_id"]
gkf = GroupKFold(n_splits=4)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(solver='lbfgs', random_state=42)
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)
y_prob = model.predict_proba(X_test_scaled)

print("Accuracy Score:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Accuracy Score: 0.9950738615925815

Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00   1956605
           1       0.67      0.34      0.45     11671

    accuracy                           1.00   1968276
   macro avg       0.83      0.67      0.72   1968276
weighted avg       0.99      1.00      0.99   1968276



In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.metrics import classification_report, f1_score, precision_score, recall_score

gkf = GroupKFold(n_splits=4)
groups = dataset['client_hash_id']

f1_scores, precision_scores, recall_scores = [], [], []

for train_idx, val_idx in gkf.split(X, y, groups=groups):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)

    model = LogisticRegression()
    model.fit(X_train_scaled, y_train)

    y_pred = model.predict(X_val_scaled)

    f1_scores.append(f1_score(y_val, y_pred, zero_division=0))
    precision_scores.append(precision_score(y_val, y_pred, zero_division=0))
    recall_scores.append(recall_score(y_val, y_pred, zero_division=0))

print(f"Mean Out-of-Fold F1-Score: {np.mean(f1_scores):.4f}")
print(f"Mean Out-of-Fold Precision: {np.mean(precision_scores):.4f}")
print(f"Mean Out-of-Fold Recall: {np.mean(recall_scores):.4f}")

Mean Out-of-Fold F1-Score: 0.4288
Mean Out-of-Fold Precision: 0.6498
Mean Out-of-Fold Recall: 0.3330


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Target & Proxy Leakage Check: CLEAN

None of the features (gsc_impressions, ga4_pageviews, ctr, ga4_sessions, gsc_avg_position) directly duplicate the target logic or predict the future.

Preprocessing Leakage Check: CLEAN

In our GroupKFold pipeline, StandardScaler.fit_transform() was strictly applied to X_train inside each fold loop, and only transform() was called on X_val. This guarantees that validation distribution stats never leaked into training.

Feature Coefficient Interpretation:

gsc_avg_position ($-6.49$): Strongly negative coefficient. As average position number grows worse/larger (e.g., from position 5 down to 80), the log-odds of being in striking distance plummet sharply.  

gsc_impressions ($+0.76$): Strong positive driver. Higher impression volume increases the probability of hitting striking distance.  

ga4_sessions ($-0.27$) & ctr ($-0.03$): Small negative weights acting as conditional suppressors/adjusters relative to impressions

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

feature_weights = pd.Series(model.coef_[0], index=X.columns).sort_values(ascending=False)
print("Feature Coefficients:")
print(feature_weights)

Feature Coefficients:
gsc_impressions     0.758453
ga4_pageviews       0.308220
ctr                -0.028267
ga4_sessions       -0.268292
gsc_avg_position   -6.494522
dtype: float64


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

Overstated Claim (Before):

    Our model accurately predicts which content pages will enter the striking distance zone.

Honest, Evidence-Backed Claim (After):

    Under client-isolated cross-validation (GroupKFold), our linear model demonstrated a modest directional signal (F1 = 0.43, Precision = 0.65). The model strongly relies on position rank and impression counts as primary decision-support filters. While precise enough for prioritizing high-intent optimization queues, its lower recall (0.33) indicates it should serve as a screening heuristic rather than a definitive predictor of page potential.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.